# Drawing Modality — Model Training v2 (HOG + sklearn + augmentation ×4)


## 0. Install Dependencies

In [11]:
import subprocess, sys

packages = [
    "numpy", "pandas", "scikit-learn", "scikit-image",
    "Pillow", "matplotlib", "requests", "scipy", "joblib", "imbalanced-learn"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

All packages ready.


## 2. Chargement et split train / test par sujet


In [12]:
from pathlib import Path
import numpy as np

REPO_ROOT  = Path('.').resolve().parent
DATA_HC    = REPO_ROOT / 'data' / 'handpd' / 'hc' / 'HealthySpiral'
DATA_PD    = REPO_ROOT / 'data' / 'handpd' / 'pd' / 'PatientSpiral'
MODEL_DIR  = REPO_ROOT / 'models'
MODEL_PATH = MODEL_DIR / 'drawing_spiral_v1_pipeline.joblib'

# Paramètres HOG — cellules 8×8 plus fines (vs 16×16 avant) → plus sensibles aux tremblements
IMG_SIZE         = (128, 128)
HOG_ORIENTATIONS = 9
HOG_PIXELS_CELL  = (8, 8)
HOG_CELLS_BLOCK  = (2, 2)

# 20 % des sujets réservés au test final (hors augmentation)
TEST_SUBJECT_FRACTION = 0.20
RANDOM_STATE = 42

print('REPO_ROOT:', REPO_ROOT)
print('HC images:', len(list(DATA_HC.glob('*.jpg'))))
print('PD images:', len(list(DATA_PD.glob('*.jpg'))))

REPO_ROOT: C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection
HC images: 140
PD images: 124


## 3. Augmentation ×4 par rotation (train uniquement)


In [13]:
import random
from PIL import Image


def load_raw(directory: Path, label: int):
    """Retourne (PIL_images, labels, subject_ids). Sujet = suffixe du nom de fichier."""
    imgs, labs, sids = [], [], []
    for path in sorted(directory.glob('*.jpg')):
        sid = path.stem.split('-')[-1]   # 'H10', 'P3', …
        imgs.append(Image.open(path).convert('L'))
        labs.append(label)
        sids.append(sid)
    return imgs, labs, sids


hc_imgs, hc_labs, hc_ids = load_raw(DATA_HC, label=0)
pd_imgs, pd_labs, pd_ids = load_raw(DATA_PD, label=1)

all_imgs = hc_imgs + pd_imgs
all_labs = hc_labs + pd_labs
all_sids = hc_ids  + pd_ids

# --- Split test par sujet ---
random.seed(RANDOM_STATE)
hc_subjects = sorted(set(hc_ids))
pd_subjects = sorted(set(pd_ids))
n_test_hc = max(1, round(len(hc_subjects) * TEST_SUBJECT_FRACTION))
n_test_pd = max(1, round(len(pd_subjects) * TEST_SUBJECT_FRACTION))
test_subjects = set(random.sample(hc_subjects, n_test_hc) + random.sample(pd_subjects, n_test_pd))

train_imgs, train_labs, train_sids = [], [], []
test_imgs,  test_labs,  test_sids  = [], [], []

for img, lab, sid in zip(all_imgs, all_labs, all_sids):
    if sid in test_subjects:
        test_imgs.append(img); test_labs.append(lab); test_sids.append(sid)
    else:
        train_imgs.append(img); train_labs.append(lab); train_sids.append(sid)

print(f'Train : {len(train_imgs)} images  ({sum(l==0 for l in train_labs)} HC / {sum(l==1 for l in train_labs)} PD)')
print(f'Test  : {len(test_imgs)}  images  ({sum(l==0 for l in test_labs)} HC / {sum(l==1 for l in test_labs)} PD)')
print(f'Sujets test : {sorted(test_subjects)}')

Train : 212 images  (112 HC / 100 PD)
Test  : 52  images  (28 HC / 24 PD)
Sujets test : ['H10', 'H13', 'H17', 'H24', 'H27', 'H32', 'H9', 'P11', 'P12', 'P25', 'P26', 'P29', 'P30']


## 4. Visualisation (exemples du train augmenté)

In [14]:
ROTATIONS = [0, 90, 180, 270]

aug_imgs, aug_labs, aug_sids = [], [], []
for img, lab, sid in zip(train_imgs, train_labs, train_sids):
    for angle in ROTATIONS:
        aug_imgs.append(img.rotate(angle, expand=False))
        aug_labs.append(lab)
        aug_sids.append(sid)   # même sujet pour les 4 rotations

print(f'Après augmentation : {len(aug_imgs)} images train')
print(f'  HC : {sum(l==0 for l in aug_labs)}  |  PD : {sum(l==1 for l in aug_labs)}')

Après augmentation : 848 images train
  HC : 448  |  PD : 400


## 4. Extraction HOG


In [15]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

hc_train_idx = [i for i, l in enumerate(aug_labs) if l == 0]
pd_train_idx = [i for i, l in enumerate(aug_labs) if l == 1]
step_hc = max(1, len(hc_train_idx) // 4)
step_pd = max(1, len(pd_train_idx) // 4)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes[0]):
    ax.imshow(aug_imgs[hc_train_idx[i * step_hc]], cmap='gray')
    ax.set_title(f'HC rot{ROTATIONS[i % 4]}°', fontsize=9)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(aug_imgs[pd_train_idx[i * step_pd]], cmap='gray')
    ax.set_title(f'PD rot{ROTATIONS[i % 4]}°', fontsize=9)
    ax.axis('off')

fig.suptitle('Exemples augmentés — train (haut : HC, bas : PD)', fontsize=12, fontweight='bold')
plt.tight_layout()
fig_path = REPO_ROOT / 'data' / 'handpd' / 'samples_augmented.png'
plt.savefig(fig_path, dpi=80)
plt.show()
print('Saved:', fig_path)

Saved: C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\data\handpd\samples_augmented.png


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_13772\2441726694.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Extraction HOG

**Prétraitement :** niveaux de gris → 128×128 → inversion (trait = clair sur fond sombre).  
Cellules 8×8 pixels (vs 16×16 avant) : plus fines, plus sensibles aux tremblements locaux.  
Cette fonction `preprocess_image` est réutilisée à l'identique dans `DrawingPredictor`.

In [16]:
from skimage.feature import hog


def preprocess_image(pil_img: Image.Image, target_size=IMG_SIZE) -> np.ndarray:
    """PIL Image → float32 numpy, trait clair sur fond sombre (inversion)."""
    img = pil_img.convert('L').resize(target_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    return 1.0 - arr   # inversion : trait = clair


def extract_hog(pil_img: Image.Image) -> np.ndarray:
    """Retourne un vecteur HOG 1-D de longueur fixe."""
    arr = preprocess_image(pil_img)
    return hog(
        arr,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_CELL,
        cells_per_block=HOG_CELLS_BLOCK,
        block_norm='L2-Hys',
        feature_vector=True,
    ).astype(np.float32)


print('Extraction HOG du jeu augmenté…')
X_aug = np.array([extract_hog(img) for img in aug_imgs])
y_aug = np.array(aug_labs)
g_aug = np.array(aug_sids)

X_test = np.array([extract_hog(img) for img in test_imgs])
y_test = np.array(test_labs)

print(f'X_aug  : {X_aug.shape}  |  X_test : {X_test.shape}')
print(f'Vecteur HOG : {X_aug.shape[1]} features')

Extraction HOG du jeu augmenté…
X_aug  : (848, 8100)  |  X_test : (52, 8100)
Vecteur HOG : 8100 features


## 6. Validation croisée par sujet (StratifiedGroupKFold)

Les 4 rotations d'un même sujet restent toujours dans le même pli → pas de fuite de données.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score


def build_pipeline() -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            random_state=RANDOM_STATE,
        )),
    ])


unique_subjects = np.unique(g_aug)
n_splits = min(5, len(unique_subjects) // 4)
cv = StratifiedGroupKFold(n_splits=n_splits)

cv_auc_scores = []
for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_aug, y_aug, g_aug)):
    pipe = build_pipeline()
    pipe.fit(X_aug[tr_idx], y_aug[tr_idx])
    proba = pipe.predict_proba(X_aug[val_idx])[:, 1]
    auc = roc_auc_score(y_aug[val_idx], proba)
    cv_auc_scores.append(auc)
    print(f'  Fold {fold_i+1}/{n_splits}  AUC={auc:.3f}')

print(f'\nAUC moyen CV : {np.mean(cv_auc_scores):.3f} ± {np.std(cv_auc_scores):.3f}')

  Fold 1/5  AUC=0.920
  Fold 2/5  AUC=0.969
  Fold 3/5  AUC=0.938
  Fold 4/5  AUC=0.919
  Fold 5/5  AUC=0.955

AUC moyen CV : 0.940 ± 0.020


## 7. Recherche du seuil de décision (OOF)

In [18]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, f1_score

y_oof = np.zeros(len(y_aug))
for tr_idx, val_idx in cv.split(X_aug, y_aug, g_aug):
    pipe_oof = build_pipeline()
    pipe_oof.fit(X_aug[tr_idx], y_aug[tr_idx])
    y_oof[val_idx] = pipe_oof.predict_proba(X_aug[val_idx])[:, 1]

fpr, tpr, thresholds = roc_curve(y_aug, y_oof)
f1_scores = [f1_score(y_aug, y_oof >= t) for t in thresholds]
best_idx  = int(np.argmax(f1_scores))
DECISION_THRESHOLD = float(thresholds[best_idx])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(fpr, tpr, color='steelblue')
ax1.plot([0, 1], [0, 1], '--', color='gray')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR'); ax1.set_title('Courbe ROC (OOF)')
ax2.plot(thresholds, f1_scores, color='darkorange')
ax2.axvline(DECISION_THRESHOLD, color='red', linestyle='--', label=f'seuil={DECISION_THRESHOLD:.2f}')
ax2.set_xlabel('Seuil'); ax2.set_ylabel('F1'); ax2.set_title('F1 vs seuil'); ax2.legend()
plt.tight_layout()
plt.savefig(REPO_ROOT / 'data' / 'handpd' / 'roc_threshold.png', dpi=80)
plt.show()
print(f'Seuil retenu : {DECISION_THRESHOLD:.3f}')

Seuil retenu : 0.380


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_13772\1439235497.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Entraînement final sur tout le train augmenté

In [19]:
final_pipeline = build_pipeline()
final_pipeline.fit(X_aug, y_aug)

print('Probabilités sur 4 exemples train :')
print(final_pipeline.predict_proba(X_aug[:4]).round(3))
print(f'\nAUC moyen (CV) : {np.mean(cv_auc_scores):.3f} ± {np.std(cv_auc_scores):.3f}')
print(f'Seuil retenu   : {DECISION_THRESHOLD:.3f}')

Probabilités sur 4 exemples train :
[[0.995 0.005]
 [0.974 0.026]
 [0.993 0.007]
 [0.982 0.018]]

AUC moyen (CV) : 0.940 ± 0.020
Seuil retenu   : 0.380


## 9. Évaluation sur le jeu de test 



In [20]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= DECISION_THRESHOLD).astype(int)

test_auc = roc_auc_score(y_test, y_test_proba)
print(f'AUC test (sujets non vus, images non augmentées) : {test_auc:.3f}')
print()
print(classification_report(y_test, y_test_pred, target_names=['HC', 'PD']))
print('Matrice de confusion (HC=0, PD=1) :')
print(confusion_matrix(y_test, y_test_pred))

AUC test (sujets non vus, images non augmentées) : 0.987

              precision    recall  f1-score   support

          HC       1.00      0.75      0.86        28
          PD       0.77      1.00      0.87        24

    accuracy                           0.87        52
   macro avg       0.89      0.88      0.86        52
weighted avg       0.90      0.87      0.86        52

Matrice de confusion (HC=0, PD=1) :
[[21  7]
 [ 0 24]]


## 10. Export de l'artefact

In [21]:
import joblib

MODEL_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    'pipeline': final_pipeline,
    'hog_params': {
        'img_size':        IMG_SIZE,
        'orientations':    HOG_ORIENTATIONS,
        'pixels_per_cell': HOG_PIXELS_CELL,
        'cells_per_block': HOG_CELLS_BLOCK,
    },
    'model':       'drawing_spiral_hog_gbc_v2',
    'threshold':   DECISION_THRESHOLD,
    'cv_auc_mean': float(np.mean(cv_auc_scores)),
    'cv_auc_std':  float(np.std(cv_auc_scores)),
    'test_auc':    float(test_auc),
    'trained_on':  'handpd_newhandpd_spirals_x4rotations',
    'note':        'NewHandPD (UNESP). HOG 8x8 px/cell, aug x4 rotations. Prototype exploratoire — pas un dispositif médical.',
}

joblib.dump(artifact, MODEL_PATH)
print(f'Modèle sauvegardé : {MODEL_PATH}')

loaded = joblib.load(MODEL_PATH)
assert loaded['hog_params'] == artifact['hog_params']
print('Vérification OK.')
print(f"AUC CV   : {loaded['cv_auc_mean']:.3f} ± {loaded['cv_auc_std']:.3f}")
print(f"AUC test : {loaded['test_auc']:.3f}")
print(f"Seuil    : {loaded['threshold']:.3f}")

Modèle sauvegardé : C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\models\drawing_spiral_v1_pipeline.joblib
Vérification OK.
AUC CV   : 0.940 ± 0.020
AUC test : 0.987
Seuil    : 0.380


## 11. Test d'inférence bout en bout



In [22]:
import io as _io

for label_name, src_imgs, src_labs in [('HC', hc_imgs[:2], [0, 0]), ('PD', pd_imgs[:2], [1, 1])]:
    for img, true_lab in zip(src_imgs, src_labs):
        buf = _io.BytesIO()
        img.save(buf, format='PNG')
        received = Image.open(_io.BytesIO(buf.getvalue()))
        feats = extract_hog(received).reshape(1, -1)
        score = loaded['pipeline'].predict_proba(feats)[0, 1]
        pred  = 'PD' if score >= loaded['threshold'] else 'HC'
        ok    = 'OK' if pred == label_name else 'ERREUR'
        print(f'[{ok}] Vraie={label_name}  Score PD={score:.3f}  Prédit={pred}')

[OK] Vraie=HC  Score PD=0.005  Prédit=HC
[ERREUR] Vraie=HC  Score PD=0.891  Prédit=PD
[OK] Vraie=PD  Score PD=0.984  Prédit=PD
[OK] Vraie=PD  Score PD=0.993  Prédit=PD
